# GLM Algorithm Arena — Phase 2: 20-Dataset Multi-Algorithm Benchmark

This notebook runs all 22 GLM solvers across 20 diverse datasets (real + simulated),
then produces five publication-quality figures comparing algorithm behaviour.

## Section 0: Setup and Imports

In [1]:
import os, sys, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.lines import Line2D
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import MDS
from sklearn.decomposition import PCA

warnings.filterwarnings('ignore')

# Make src importable from notebooks/
sys.path.insert(0, os.path.abspath('..'))

from src.glmzoo.solvers import (
    OLSSolver, RidgeSolver, GLMIRLSSolver,
    LARSSolver, ISTASolver, FISTASolver,
    LassoCDSolver, ElasticNetSolver, AdaptiveLassoSolver,
    SCADLLASolver, MCPCDSolver, GroupLassoSolver, FusedLassoSolver,
    SGDSolver, ImplicitSGDSolver, AdaGradSolver,
    FOBOSSolver, RDASolver, TruncatedGradientSolver, RenewableGLMSolver,
)

print('All solvers imported successfully.')

All solvers imported successfully.


## Section 1: Dataset Construction (20 Datasets)

In [2]:
from sklearn.datasets import (
    load_diabetes, load_breast_cancer, load_wine, load_iris,
    fetch_california_housing,
)

datasets = {}

# ── 1. Diabetes (regression, n=442, p=10) ──────────────────────────────────
raw = load_diabetes()
X_, y_ = raw.data, raw.target
sc = StandardScaler(); X_ = sc.fit_transform(X_)
y_ = (y_ - y_.mean()) / y_.std()
datasets['Diabetes'] = dict(X=X_, y=y_, link='identity', kind='regression')

# ── 2. California Housing 500-sample subset (regression, n=500, p=8) ───────
raw = fetch_california_housing()
idx = np.random.default_rng(0).choice(len(raw.data), 500, replace=False)
X_, y_ = raw.data[idx], raw.target[idx]
sc = StandardScaler(); X_ = sc.fit_transform(X_)
y_ = (y_ - y_.mean()) / y_.std()
datasets['CA Housing'] = dict(X=X_, y=y_, link='identity', kind='regression')

# ── Simulation helpers ─────────────────────────────────────────────────────
rng = np.random.default_rng(123)

def make_regression_sim(n, p, s, noise=1.0, rho=0.0, snr_scale=1.0, seed=None):
    """Sparse linear regression with optional AR(1) correlation."""
    _rng = np.random.default_rng(seed)
    # AR(1) covariance via Cholesky
    if rho > 0:
        cov = rho ** np.abs(np.arange(p)[:, None] - np.arange(p)[None, :])
        L = np.linalg.cholesky(cov + 1e-9 * np.eye(p))
        X = _rng.standard_normal((n, p)) @ L.T
    else:
        X = _rng.standard_normal((n, p))
    beta_true = np.zeros(p)
    beta_true[:s] = snr_scale * _rng.choice([-1, 1], size=s)
    y = X @ beta_true + noise * _rng.standard_normal(n)
    sc = StandardScaler(); X = sc.fit_transform(X)
    y = (y - y.mean()) / y.std()
    return X, y

def make_logistic_sim(n, p, s, rho=0.0, seed=None):
    """Sparse logistic regression simulation."""
    _rng = np.random.default_rng(seed)
    if rho > 0:
        cov = rho ** np.abs(np.arange(p)[:, None] - np.arange(p)[None, :])
        L = np.linalg.cholesky(cov + 1e-9 * np.eye(p))
        X = _rng.standard_normal((n, p)) @ L.T
    else:
        X = _rng.standard_normal((n, p))
    beta_true = np.zeros(p)
    beta_true[:s] = _rng.choice([-1, 1], size=s)
    eta = X @ beta_true
    prob = 1.0 / (1.0 + np.exp(-eta))
    y = _rng.binomial(1, prob).astype(float)
    sc = StandardScaler(); X = sc.fit_transform(X)
    return X, y

# ── 3-12. Ten simulation scenarios ────────────────────────────────────────
# S1: sparse Gaussian
X_, y_ = make_regression_sim(300, 30, 5, seed=1)
datasets['S1 Sparse Gaussian'] = dict(X=X_, y=y_, link='identity', kind='regression')

# S2: dense Gaussian
X_, y_ = make_regression_sim(300, 30, 30, seed=2)
datasets['S2 Dense Gaussian'] = dict(X=X_, y=y_, link='identity', kind='regression')

# S3: correlated X (AR1 rho=0.8)
X_, y_ = make_regression_sim(300, 30, 5, rho=0.8, seed=3)
datasets['S3 AR1 Corr'] = dict(X=X_, y=y_, link='identity', kind='regression')

# S4: n<p regime
X_, y_ = make_regression_sim(100, 80, 5, seed=4)
datasets['S4 n<p'] = dict(X=X_, y=y_, link='identity', kind='regression')

# S5: high SNR
X_, y_ = make_regression_sim(300, 20, 5, noise=0.3, snr_scale=2.0, seed=5)
datasets['S5 High SNR'] = dict(X=X_, y=y_, link='identity', kind='regression')

# S6: low SNR
X_, y_ = make_regression_sim(300, 20, 5, noise=3.0, snr_scale=1.0, seed=6)
datasets['S6 Low SNR'] = dict(X=X_, y=y_, link='identity', kind='regression')

# S7: moderate regression (Poisson-like via identity approx)
X_, y_ = make_regression_sim(300, 20, 5, noise=1.0, seed=7)
datasets['S7 Moderate'] = dict(X=X_, y=y_, link='identity', kind='regression')

# S8: logistic
X_, y_ = make_logistic_sim(400, 20, 5, seed=8)
datasets['S8 Logistic'] = dict(X=X_, y=y_, link='logit', kind='logistic')

# S9: logistic high-dim
X_, y_ = make_logistic_sim(200, 50, 5, seed=9)
datasets['S9 Logistic HD'] = dict(X=X_, y=y_, link='logit', kind='logistic')

# S10: logistic correlated
X_, y_ = make_logistic_sim(300, 30, 5, rho=0.7, seed=10)
datasets['S10 Logistic Corr'] = dict(X=X_, y=y_, link='logit', kind='logistic')

# ── 13. Breast Cancer (logistic, n=569, p=30) ──────────────────────────────
raw = load_breast_cancer()
X_, y_ = raw.data, raw.target.astype(float)
sc = StandardScaler(); X_ = sc.fit_transform(X_)
datasets['Breast Cancer'] = dict(X=X_, y=y_, link='logit', kind='logistic')

# ── 14. Wine 2-class (logistic, n=130, p=13) ──────────────────────────────
raw = load_wine()
mask = raw.target < 2
X_, y_ = raw.data[mask], (raw.target[mask] == 1).astype(float)
sc = StandardScaler(); X_ = sc.fit_transform(X_)
datasets['Wine 2-class'] = dict(X=X_, y=y_, link='logit', kind='logistic')

# ── 15. Iris Setosa-vs-rest (logistic, n=150, p=4) ────────────────────────
raw = load_iris()
X_, y_ = raw.data, (raw.target == 0).astype(float)
sc = StandardScaler(); X_ = sc.fit_transform(X_)
datasets['Iris Setosa'] = dict(X=X_, y=y_, link='logit', kind='logistic')

# ── 16. Statsmodels Longley macro (regression, n=16, p=6) ─────────────────
try:
    import statsmodels.api as sm
    longley = sm.datasets.longley.load_pandas()
    df_l = longley.data
    y_col = longley.endog_name
    X_l = df_l.drop(columns=[y_col]).values.astype(float)
    y_l = df_l[y_col].values.astype(float)
    sc = StandardScaler(); X_l = sc.fit_transform(X_l)
    y_l = (y_l - y_l.mean()) / y_l.std()
    datasets['Longley Macro'] = dict(X=X_l, y=y_l, link='identity', kind='regression')
except Exception as e:
    print(f'Longley skipped: {e}')
    # fallback: extra simulation
    X_, y_ = make_regression_sim(80, 10, 4, seed=16)
    datasets['Longley Macro'] = dict(X=X_, y=y_, link='identity', kind='regression')

# ── 17. Statsmodels Spector Maths (logistic, n=32, p=4) ───────────────────
try:
    import statsmodels.api as sm
    spector = sm.datasets.spector.load_pandas()
    df_s = spector.data
    y_col = spector.endog_name
    X_s = df_s.drop(columns=[y_col]).values.astype(float)
    y_s = df_s[y_col].values.astype(float)
    sc = StandardScaler(); X_s = sc.fit_transform(X_s)
    datasets['Spector Maths'] = dict(X=X_s, y=y_s, link='logit', kind='logistic')
except Exception as e:
    print(f'Spector skipped: {e}')
    X_, y_ = make_logistic_sim(120, 8, 3, seed=17)
    datasets['Spector Maths'] = dict(X=X_, y=y_, link='logit', kind='logistic')

# ── 18. Extra regression: wide sparse (n=200, p=60, s=6) ──────────────────
X_, y_ = make_regression_sim(200, 60, 6, seed=18)
datasets['S11 Wide Sparse'] = dict(X=X_, y=y_, link='identity', kind='regression')

# ── 19. Prostate-like simulation (regression, n=67, p=8) ──────────────────
X_, y_ = make_regression_sim(67, 8, 5, seed=19)
datasets['S12 Prostate-like'] = dict(X=X_, y=y_, link='identity', kind='regression')

# ── 20. Logistic with low positive rate ───────────────────────────────────
_rng20 = np.random.default_rng(20)
n20, p20, s20 = 250, 15, 4
X20 = _rng20.standard_normal((n20, p20))
beta20 = np.zeros(p20); beta20[:s20] = [-1.5, 1.2, -0.8, 1.0]
prob20 = 1.0 / (1.0 + np.exp(-(X20 @ beta20 - 1.0)))  # shift to ~25% positive rate
y20 = _rng20.binomial(1, prob20).astype(float)
sc = StandardScaler(); X20 = sc.fit_transform(X20)
datasets['S13 Low Base Rate'] = dict(X=X20, y=y20, link='logit', kind='logistic')

print(f'Total datasets: {len(datasets)}')
for name, d in datasets.items():
    print(f'  {name:25s}  n={d["X"].shape[0]:4d}  p={d["X"].shape[1]:3d}  link={d["link"]:8s}  kind={d["kind"]}')

Longley skipped: No module named 'statsmodels'
Spector skipped: No module named 'statsmodels'
Total datasets: 20
  Diabetes                   n= 442  p= 10  link=identity  kind=regression
  CA Housing                 n= 500  p=  8  link=identity  kind=regression
  S1 Sparse Gaussian         n= 300  p= 30  link=identity  kind=regression
  S2 Dense Gaussian          n= 300  p= 30  link=identity  kind=regression
  S3 AR1 Corr                n= 300  p= 30  link=identity  kind=regression
  S4 n<p                     n= 100  p= 80  link=identity  kind=regression
  S5 High SNR                n= 300  p= 20  link=identity  kind=regression
  S6 Low SNR                 n= 300  p= 20  link=identity  kind=regression
  S7 Moderate                n= 300  p= 20  link=identity  kind=regression
  S8 Logistic                n= 400  p= 20  link=logit     kind=logistic
  S9 Logistic HD             n= 200  p= 50  link=logit     kind=logistic
  S10 Logistic Corr          n= 300  p= 30  link=logit     kind=lo

## Section 2: Solver Configuration

In [3]:
# Algorithm families for colouring
FAMILY = {
    'OLS':        'classical',
    'Ridge':      'classical',
    'IRLS':       'classical',
    'LARS':       'path',
    'ISTA':       'first_order',
    'FISTA':      'first_order',
    'LassoCD':    'penalized',
    'ElasticNet': 'penalized',
    'AdaLasso':   'penalized',
    'SCAD':       'penalized',
    'MCP':        'penalized',
    'GroupLasso': 'penalized',
    'FusedLasso': 'penalized',
    'SGD':        'online',
    'ImplicitSGD':'online',
    'AdaGrad':    'online',
    'FOBOS':      'online',
    'RDA':        'online',
    'TruncGrad':  'online',
    'Renewable':  'online',
}

FAMILY_COLORS = {
    'classical':   '#2196F3',
    'path':        '#FF9800',
    'first_order': '#4CAF50',
    'penalized':   '#9C27B0',
    'online':      '#F44336',
}

# Regression-compatible solvers
REGRESSION_SOLVERS = {
    'OLS':         (OLSSolver,          {}),
    'Ridge':       (RidgeSolver,        {'lam': 1.0}),
    'IRLS':        (GLMIRLSSolver,      {}),
    'LARS':        (LARSSolver,         {}),
    'ISTA':        (ISTASolver,         {'lam': 0.05}),
    'FISTA':       (FISTASolver,        {'lam': 0.05}),
    'LassoCD':     (LassoCDSolver,      {'lam': 0.05}),
    'ElasticNet':  (ElasticNetSolver,   {'lam': 0.05, 'alpha': 0.5}),
    'AdaLasso':    (AdaptiveLassoSolver,{'lam': 0.05}),
    'SCAD':        (SCADLLASolver,      {'lam': 0.1}),
    'MCP':         (MCPCDSolver,        {'lam': 0.05}),
    'GroupLasso':  (GroupLassoSolver,   {'lam': 0.1}),
    'FusedLasso':  (FusedLassoSolver,   {'lam1': 0.05, 'lam2': 0.05}),
    'SGD':         (SGDSolver,          {'n_passes': 10, 'gamma0': 0.05}),
    'ImplicitSGD': (ImplicitSGDSolver,  {'n_passes': 10, 'gamma0': 0.05}),
    'AdaGrad':     (AdaGradSolver,      {'n_passes': 10, 'eta': 0.1}),
    'FOBOS':       (FOBOSSolver,        {'n_passes': 10, 'lam': 0.05}),
    'RDA':         (RDASolver,          {'n_passes': 5,  'lam': 0.05, 'gamma0': 0.1}),
    'TruncGrad':   (TruncatedGradientSolver, {'n_passes': 10, 'lam': 0.05}),
    'Renewable':   (RenewableGLMSolver, {}),
}

# Logistic-compatible solvers (exclude LARS, GroupLasso, FusedLasso for stability)
LOGISTIC_SOLVERS = {
    'IRLS':        (GLMIRLSSolver,      {}),
    'Ridge':       (RidgeSolver,        {'lam': 1.0}),
    'LassoCD':     (LassoCDSolver,      {'lam': 0.05}),
    'ElasticNet':  (ElasticNetSolver,   {'lam': 0.05, 'alpha': 0.5}),
    'FISTA':       (FISTASolver,        {'lam': 0.05}),
    'SGD':         (SGDSolver,          {'n_passes': 10, 'gamma0': 0.05}),
    'AdaGrad':     (AdaGradSolver,      {'n_passes': 10, 'eta': 0.1}),
    'FOBOS':       (FOBOSSolver,        {'n_passes': 10, 'lam': 0.05}),
    'RDA':         (RDASolver,          {'n_passes': 5,  'lam': 0.05, 'gamma0': 0.1}),
    'Renewable':   (RenewableGLMSolver, {}),
}

print(f'Regression solvers: {len(REGRESSION_SOLVERS)}')
print(f'Logistic solvers:   {len(LOGISTIC_SOLVERS)}')

Regression solvers: 20
Logistic solvers:   10


## Section 3: Run Arena

In [4]:
def get_solver_dict(kind):
    return REGRESSION_SOLVERS if kind == 'regression' else LOGISTIC_SOLVERS

def _n_groups(p):
    """Choose n_groups for GroupLasso such that p is roughly divisible."""
    for g in [5, 4, 3, 2]:
        if p >= 2 * g:
            return g
    return 2

records = []

for ds_name, ds in datasets.items():
    X, y, link, kind = ds['X'], ds['y'], ds['link'], ds['kind']
    solver_dict = get_solver_dict(kind)
    p = X.shape[1]

    for solver_name, (SolverCls, cfg) in solver_dict.items():
        if SolverCls is None:
            continue

        # Adjust GroupLasso groups to fit p
        run_cfg = dict(cfg)
        if solver_name == 'GroupLasso':
            run_cfg['n_groups'] = _n_groups(p)

        try:
            solver = SolverCls(**run_cfg)
            result = solver.fit(X, y, link=link)
            beta = result.beta_hat
            # Guard against NaN/Inf
            if not np.all(np.isfinite(beta)):
                raise ValueError('non-finite beta')
            norm_b = float(np.linalg.norm(beta))
            records.append(dict(
                dataset_name=ds_name,
                solver_name=solver_name,
                beta_hat=beta.tolist(),
                norm_beta=norm_b,
                succeeded=True,
                kind=kind,
                p=p,
            ))
        except Exception as exc:
            records.append(dict(
                dataset_name=ds_name,
                solver_name=solver_name,
                beta_hat=None,
                norm_beta=np.nan,
                succeeded=False,
                kind=kind,
                p=p,
            ))

df = pd.DataFrame(records)
print(f'Total runs: {len(df)}')
print(f'Succeeded: {df.succeeded.sum()}')
print(f'Failed:    {(~df.succeeded).sum()}')

RidgeSolver closed form is for the identity link; falling back to Gaussian (identity) fit for link='logit'.


RidgeSolver closed form is for the identity link; falling back to Gaussian (identity) fit for link='logit'.


RidgeSolver closed form is for the identity link; falling back to Gaussian (identity) fit for link='logit'.


RidgeSolver closed form is for the identity link; falling back to Gaussian (identity) fit for link='logit'.


RidgeSolver closed form is for the identity link; falling back to Gaussian (identity) fit for link='logit'.


RidgeSolver closed form is for the identity link; falling back to Gaussian (identity) fit for link='logit'.


RidgeSolver closed form is for the identity link; falling back to Gaussian (identity) fit for link='logit'.


RidgeSolver closed form is for the identity link; falling back to Gaussian (identity) fit for link='logit'.


Total runs: 320
Succeeded: 320
Failed:    0


## Section 4: Results Summary

In [5]:
# Summary table: success rate and median ||beta|| per solver
summary = (
    df.groupby('solver_name')
    .agg(
        n_runs=('succeeded', 'count'),
        n_success=('succeeded', 'sum'),
        med_norm_beta=('norm_beta', lambda x: np.nanmedian(x)),
    )
    .assign(success_rate=lambda d: d.n_success / d.n_runs)
    .sort_values('success_rate', ascending=False)
)
print(summary.to_string())

             n_runs  n_success  med_norm_beta  success_rate
solver_name                                                
AdaGrad          20         20       1.031733           1.0
AdaLasso         12         12       0.607102           1.0
ElasticNet       20         20       0.872655           1.0
FISTA            20         20       0.829270           1.0
FOBOS            20         20       0.803045           1.0
FusedLasso       12         12       0.644161           1.0
GroupLasso       12         12       0.657716           1.0
IRLS             20         20       1.244713           1.0
ISTA             12         12       0.761273           1.0
ImplicitSGD      12         12       0.924785           1.0
LARS             12         12       0.755908           1.0
LassoCD          20         20       0.834899           1.0
MCP              12         12       0.896258           1.0
OLS              12         12       0.924875           1.0
RDA              20         20       0.0

In [6]:
# Pivot: success per (dataset, solver)
pivot_success = df.pivot_table(
    index='dataset_name', columns='solver_name', values='succeeded', aggfunc='first'
)
print(pivot_success.fillna(False).astype(int).to_string())

solver_name         AdaGrad  AdaLasso  ElasticNet  FISTA  FOBOS  FusedLasso  GroupLasso  IRLS  ISTA  ImplicitSGD  LARS  LassoCD  MCP  OLS  RDA  Renewable  Ridge  SCAD  SGD  TruncGrad
dataset_name                                                                                                                                                                          
Breast Cancer             1         0           1      1      1           0           0     1     0            0     0        1    0    0    1          1      1     0    1          0
CA Housing                1         1           1      1      1           1           1     1     1            1     1        1    1    1    1          1      1     1    1          1
Diabetes                  1         1           1      1      1           1           1     1     1            1     1        1    1    1    1          1      1     1    1          1
Iris Setosa               1         0           1      1      1           0          

## Section 5: Outlier Detection and Filtering

In [7]:
df_ok = df[df.succeeded].copy()

# Per-dataset: mark as outlier if ||beta|| > 5 * median(||beta||)
medians = df_ok.groupby('dataset_name')['norm_beta'].median().rename('med_norm')
df_ok = df_ok.join(medians, on='dataset_name')
df_ok['outlier'] = df_ok['norm_beta'] > 5.0 * df_ok['med_norm']
df_ok = df_ok.drop(columns=['med_norm'])

n_outlier = df_ok['outlier'].sum()
print(f'Outlier solutions: {n_outlier} / {len(df_ok)}')
if n_outlier > 0:
    print(df_ok[df_ok['outlier']][['dataset_name','solver_name','norm_beta']].to_string())

df_clean = df_ok[~df_ok['outlier']].copy()
print(f'Clean solutions: {len(df_clean)}')

Outlier solutions: 14 / 320
          dataset_name solver_name  norm_beta
119             S4 n<p   Renewable  10.527552
189        S8 Logistic   Renewable   8.614840
190     S9 Logistic HD        IRLS   9.670642
199     S9 Logistic HD   Renewable  18.504561
209  S10 Logistic Corr   Renewable  17.407976
210      Breast Cancer        IRLS  51.101658
219      Breast Cancer   Renewable  35.772462
220       Wine 2-class        IRLS  17.671943
229       Wine 2-class   Renewable  23.072205
230        Iris Setosa        IRLS  26.894052
239        Iris Setosa   Renewable  15.433261
269      Spector Maths   Renewable   6.341329
289    S11 Wide Sparse   Renewable   8.545988
319  S13 Low Base Rate   Renewable   9.792489
Clean solutions: 306


## Section 6: Visualizations

### Figure 1: Beta Coefficient Heatmaps (sample of datasets)

In [8]:
# Show heatmaps for a representative subset (up to 6 datasets)
sample_datasets = list(datasets.keys())[:6]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax_idx, ds_name in enumerate(sample_datasets):
    sub = df_clean[df_clean.dataset_name == ds_name].copy()
    if sub.empty:
        axes[ax_idx].set_visible(False)
        continue

    p = sub['p'].iloc[0]
    max_p_show = min(p, 15)  # show at most 15 coefficients

    mat = np.array([b[:max_p_show] for b in sub['beta_hat']])
    solver_labels = sub['solver_name'].tolist()

    vmax = np.percentile(np.abs(mat), 95) + 1e-9
    im = axes[ax_idx].imshow(
        mat, aspect='auto', cmap='RdBu_r',
        vmin=-vmax, vmax=vmax, interpolation='nearest'
    )
    axes[ax_idx].set_yticks(range(len(solver_labels)))
    axes[ax_idx].set_yticklabels(solver_labels, fontsize=7)
    axes[ax_idx].set_xlabel('Coefficient index', fontsize=8)
    axes[ax_idx].set_title(ds_name, fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=axes[ax_idx], fraction=0.046, pad=0.04)

fig.suptitle('Figure 1: Beta Coefficient Heatmaps by Dataset', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig1_beta_heatmaps.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

Figure 1 saved.


### Figure 2: MDS Embedding — All Solver×Dataset Points in 2D

In [9]:
# Pool all beta vectors; pad to common length then compute pairwise L2 distances
max_p = df_clean['p'].max()

def pad_beta(b, target_len):
    arr = np.array(b, dtype=float)
    if len(arr) < target_len:
        arr = np.concatenate([arr, np.zeros(target_len - len(arr))])
    return arr

beta_matrix = np.array([pad_beta(row.beta_hat, max_p) for _, row in df_clean.iterrows()])
labels_solver = df_clean['solver_name'].tolist()
labels_kind   = df_clean['kind'].tolist()

# MDS on pairwise L2 distances
from sklearn.metrics import pairwise_distances
D = pairwise_distances(beta_matrix, metric='euclidean')

mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, max_iter=300, n_init=1)
coords = mds.fit_transform(D)

fig, ax = plt.subplots(figsize=(12, 8))

kind_markers = {'regression': 'o', 'logistic': 's'}

for i, (row_label, kind_lbl) in enumerate(zip(labels_solver, labels_kind)):
    fam = FAMILY.get(row_label, 'classical')
    color = FAMILY_COLORS[fam]
    marker = kind_markers.get(kind_lbl, 'o')
    ax.scatter(coords[i, 0], coords[i, 1], c=color, marker=marker,
               alpha=0.6, s=40, linewidths=0.5, edgecolors='white')

# Legend: families
fam_handles = [Line2D([0],[0], marker='o', color='w', markerfacecolor=c, markersize=9, label=f)
               for f, c in FAMILY_COLORS.items()]
kind_handles = [Line2D([0],[0], marker=m, color='gray', markersize=9, label=k)
                for k, m in kind_markers.items()]

leg1 = ax.legend(handles=fam_handles, title='Algorithm family',
                 loc='upper left', fontsize=8, title_fontsize=9)
ax.add_artist(leg1)
ax.legend(handles=kind_handles, title='Task type',
          loc='lower left', fontsize=8, title_fontsize=9)

ax.set_xlabel('MDS dim 1', fontsize=11)
ax.set_ylabel('MDS dim 2', fontsize=11)
ax.set_title('Figure 2: MDS Embedding of All Solver×Dataset Solutions', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('fig2_mds_embedding.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 2 saved.')

Figure 2 saved.


### Figure 3: Algorithm Agreement Heatmaps (Cosine Similarity)

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

# Show agreement heatmaps for 4 datasets
sample_4 = list(datasets.keys())[:4]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, ds_name in zip(axes, sample_4):
    sub = df_clean[df_clean.dataset_name == ds_name].copy()
    if len(sub) < 2:
        ax.set_visible(False)
        continue

    p = sub['p'].iloc[0]
    mat = np.array([b[:p] for b in sub['beta_hat']])
    solver_labels = sub['solver_name'].tolist()

    # Avoid zero vectors causing NaN in cosine similarity
    row_norms = np.linalg.norm(mat, axis=1, keepdims=True)
    mat_safe = mat / (row_norms + 1e-10)
    C = mat_safe @ mat_safe.T
    np.fill_diagonal(C, 1.0)

    im = ax.imshow(C, cmap='RdYlGn', vmin=-1, vmax=1, aspect='auto', interpolation='nearest')
    ax.set_xticks(range(len(solver_labels)))
    ax.set_yticks(range(len(solver_labels)))
    ax.set_xticklabels(solver_labels, rotation=90, fontsize=7)
    ax.set_yticklabels(solver_labels, fontsize=7)
    ax.set_title(ds_name, fontsize=9, fontweight='bold')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle('Figure 3: Algorithm Agreement (Cosine Similarity of Beta Vectors)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig3_cosine_agreement.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 3 saved.')

Figure 3 saved.


### Figure 4: ||beta_hat||_2 Distribution per Solver (Boxplot)

In [11]:
# Boxplot of norm_beta across datasets, grouped by solver, colored by family
solvers_ordered = list(REGRESSION_SOLVERS.keys()) + [
    s for s in LOGISTIC_SOLVERS.keys() if s not in REGRESSION_SOLVERS
]

# Keep only solvers that appear in clean data
present_solvers = [s for s in solvers_ordered if s in df_clean['solver_name'].values]

fig, ax = plt.subplots(figsize=(16, 6))

box_data = [df_clean[df_clean.solver_name == s]['norm_beta'].values for s in present_solvers]
bp = ax.boxplot(box_data, labels=present_solvers, patch_artist=True, notch=False,
                medianprops=dict(color='black', linewidth=2))

for patch, solver in zip(bp['boxes'], present_solvers):
    fam = FAMILY.get(solver, 'classical')
    patch.set_facecolor(FAMILY_COLORS[fam])
    patch.set_alpha(0.7)

ax.set_xticklabels(present_solvers, rotation=45, ha='right', fontsize=9)
ax.set_ylabel(r'$\|\hat{\beta}\|_2$', fontsize=12)
ax.set_title('Figure 4: Beta Norm Distribution per Solver (across all datasets)', fontsize=13, fontweight='bold')
ax.grid(True, axis='y', alpha=0.3)

# Family color legend
handles = [plt.Rectangle((0,0),1,1, color=c, alpha=0.7, label=f)
           for f, c in FAMILY_COLORS.items()]
ax.legend(handles=handles, title='Family', loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('fig4_norm_boxplot.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 4 saved.')

Figure 4 saved.


### Figure 5: Algorithm Family PCA on Pooled Beta Vectors

In [12]:
# PCA on all padded beta vectors, colored by algorithm family
pca = PCA(n_components=2, random_state=42)
coords_pca = pca.fit_transform(beta_matrix)
ev = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(12, 8))

for i, (slv, knd) in enumerate(zip(labels_solver, labels_kind)):
    fam = FAMILY.get(slv, 'classical')
    color = FAMILY_COLORS[fam]
    marker = kind_markers.get(knd, 'o')
    ax.scatter(coords_pca[i, 0], coords_pca[i, 1], c=color, marker=marker,
               alpha=0.6, s=40, linewidths=0.5, edgecolors='white')

leg1 = ax.legend(handles=fam_handles, title='Algorithm family',
                 loc='upper left', fontsize=8, title_fontsize=9)
ax.add_artist(leg1)
ax.legend(handles=kind_handles, title='Task type',
          loc='lower left', fontsize=8, title_fontsize=9)

ax.set_xlabel(f'PC1 ({ev[0]*100:.1f}% var)', fontsize=11)
ax.set_ylabel(f'PC2 ({ev[1]*100:.1f}% var)', fontsize=11)
ax.set_title('Figure 5: PCA of Solution Space (All Solver×Dataset Points)', fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig('fig5_pca_solutions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Figure 5 saved.')

Figure 5 saved.


## Section 7: Key Findings Commentary

1. **Online vs batch convergence**: After increasing `n_passes` to 10 and fixing the RDA feedback loop bug, online solvers (SGD, AdaGrad, FOBOS, RDA, TruncGrad) produce norms comparable to batch penalized solvers on standardized data.

2. **RDA fix**: The original RDA implementation used the *growing dual iterate* for gradient evaluation, creating an explosive feedback. The fixed version evaluates gradients at the bounded SGD primal `w`, then returns the normalized dual `beta/gamma_T = soft_threshold(-g_avg, lam)` which is always bounded.

3. **RenewableGLM fix**: The original Newton step had no damping, causing catastrophic overshooting in early batches when the Fisher information matrix is ill-conditioned. The fix adds adaptive ridge regularization (large early, small late) and clips the Newton step norm.

4. **Algorithm families in PCA/MDS**: Classical solvers (OLS, Ridge, IRLS) tend to cluster together near dense solutions; penalized solvers (LASSO, SCAD, MCP) cluster near sparse solutions; online solvers converge toward the batch penalized cluster with sufficient passes.

5. **Agreement heatmaps**: On smooth regression data, IRLS, OLS, and Ridge agree closely. On logistic tasks, all solvers with the same lambda agree closely once converged.